## Prerequisites

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import numpy as np
from tqdm import tqdm
import scipy.io as sio
import h5py
import json
import matplotlib.pyplot as plt
import pandas as pd

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

## Data Augmentation

In [ ]:
class SegmentationAugmentation:
    # Adjusted defaults for SIFT Flow (upscaled)
    def __init__(self, base_size=512, crop_size=480, scale_range=(0.7, 1.3)):
        self.base_size = base_size
        self.crop_size = crop_size
        self.scale_range = scale_range
        
        # Color augmentation
        self.color_jitter = transforms.ColorJitter(
            brightness=0.4,
            contrast=0.4,
            saturation=0.4,
            hue=0.1
        )
    
    def __call__(self, image, label):
        # Random scaling
        scale = np.random.uniform(self.scale_range[0], self.scale_range[1])
        target_size = int(self.base_size * scale)
        
        image = image.resize((target_size, target_size), Image.BILINEAR)
        label = label.resize((target_size, target_size), Image.NEAREST)
        
        # Random crop
        w, h = image.size
        if w > self.crop_size and h > self.crop_size:
            x1 = np.random.randint(0, w - self.crop_size)
            y1 = np.random.randint(0, h - self.crop_size)
            image = image.crop((x1, y1, x1 + self.crop_size, y1 + self.crop_size))
            label = label.crop((x1, y1, x1 + self.crop_size, y1 + self.crop_size))
        else:
            # Pad if too small
            image = image.resize((self.crop_size, self.crop_size), Image.BILINEAR)
            label = label.resize((self.crop_size, self.crop_size), Image.NEAREST)
        
        # Random horizontal flip
        if np.random.random() > 0.5:
            image = image.transpose(Image.FLIP_LEFT_RIGHT)
            label = label.transpose(Image.FLIP_LEFT_RIGHT)
        
        # Color jittering (only on image)
        if np.random.random() > 0.5:
            image = self.color_jitter(image)
        
        return image, label

class TestAugmentation:
    def __init__(self, size=(512, 512)):
        self.size = size
    
    def __call__(self, image, label):
        image = image.resize(self.size, Image.BILINEAR)
        label = label.resize(self.size, Image.NEAREST)
        return image, label

## Dataset processing

In [ ]:
class MatSiftDataset(Dataset):
    def __init__(self, mat_path, split='train', transform=None, augmentation=None):
        self.transform = transform
        self.augmentation = augmentation
        self.split = split
        self.upscale_factor = 2.0
        
        if not os.path.exists(mat_path):
            raise FileNotFoundError(f"File not found: {mat_path}")

        # Load MAT file
        try:
            mat_data = sio.loadmat(mat_path)
        except NotImplementedError:
            mat_data = h5py.File(mat_path, 'r')

        # Extract Images and Labels
        if 'Images' in mat_data: raw_imgs = mat_data['Images']
        elif 'imgs' in mat_data: raw_imgs = mat_data['imgs']
        elif 'SiftFlowData' in mat_data: raw_imgs = mat_data['SiftFlowData'] # Common struct
        else: raise KeyError(f"Keys found: {list(mat_data.keys())}")

        if 'Label' in mat_data: raw_lbls = mat_data['Label']
        elif 'categories' in mat_data: raw_lbls = mat_data['categories']
        elif 'class' in mat_data: raw_lbls = mat_data['class']
        else: raise KeyError("Could not find labels.")

        # Standardize Dimensions (N, H, W, C)
        if raw_imgs.ndim == 4 and raw_imgs.shape[3] > raw_imgs.shape[0]:
            self.images = np.transpose(raw_imgs, (3, 0, 1, 2))
            if raw_lbls.ndim == 3: self.labels = np.transpose(raw_lbls, (2, 0, 1))
            elif raw_lbls.ndim == 4: self.labels = np.transpose(raw_lbls, (3, 0, 1, 2)).squeeze()
        else:
            self.images = raw_imgs
            self.labels = raw_lbls

        # Split
        split_point = 2488
        total = len(self.images)
        if split == 'train': self.indices = np.arange(0, min(split_point, total))
        else: self.indices = np.arange(min(split_point, total), total)

        # Setup Label Shifting
        self.num_classes = 33
        
        # Check min label in the whole dataset to decide offset
        sample_lbl = self.labels[0]
        if np.min(sample_lbl) >= 1:
            self.offset = 1 
            print(f"Info: Detected 1-based labels. Will shift by -1.")
        else:
            self.offset = 0
            print(f"Info: Detected 0-based labels.")

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        
        img_data = self.images[real_idx]
        lbl_data = self.labels[real_idx]
        
        # Image to uint8
        if img_data.max() <= 1.0: img_data = (img_data * 255).astype(np.uint8)
        else: img_data = img_data.astype(np.uint8)
        
        # Label processing
        # 1. Shift
        lbl_data = lbl_data.astype(np.int32) - self.offset
        
        # 2. PIL Conversion
        rgb = Image.fromarray(img_data)
        semantic = Image.fromarray(lbl_data.astype(np.uint8)) # careful: negative values wrap here
        
        # 3. Upscale
        w, h = rgb.size
        target_w, target_h = int(w * self.upscale_factor), int(h * self.upscale_factor)
        rgb = rgb.resize((target_w, target_h), Image.BILINEAR)
        semantic = semantic.resize((target_w, target_h), Image.NEAREST)

        # 4. Augmentation
        if self.augmentation is not None:
            rgb, semantic = self.augmentation(rgb, semantic)
        
        # 5. Transform Image
        if self.transform:
            rgb = self.transform(rgb)
        else:
            rgb = transforms.ToTensor()(rgb)
        
        # 6. Final Label Safety Check (CRITICAL FIX)
        label_np = np.array(semantic, dtype=np.int64)
        
        # Invalid mask checking
        mask_invalid = (label_np < 0) | (label_np >= self.num_classes)
        label_np[mask_invalid] = 255  # Set to Ignore Index
        
        label = torch.from_numpy(label_np).long()
        
        return rgb.float(), label

def verify_dataset_range(dataset, name="Train"):
    print(f"Verifying {name} dataset labels")
    min_val = 9999
    max_val = -9999
    
    # Check the first 50 images
    for i in range(min(50, len(dataset))):
        _, label = dataset[i]
        # Ignore 255 for min/max calculation
        valid_mask = label != 255
        if valid_mask.sum() > 0:
            curr_min = label[valid_mask].min().item()
            curr_max = label[valid_mask].max().item()
            min_val = min(min_val, curr_min)
            max_val = max(max_val, curr_max)
            
    print(f"   Range found: [{min_val}, {max_val}]")
    
    if max_val >= 33:
        print("Found labels >= 33")
    if min_val < 0:
        print("Found labels < 0")

## Model architecture classes

In [ ]:
class SpatialPyramidPooling(nn.Module):
    def __init__(self, pool_sizes=[1, 2, 4]):
        super().__init__()
        self.pool_sizes = pool_sizes
    
    def forward(self, x):
        B, C, H, W = x.shape
        pooled = [x]
        for ps in self.pool_sizes:
            pooled_feat = F.adaptive_max_pool2d(x, (ps, ps))
            upsampled = F.interpolate(pooled_feat, size=(H, W), mode='bilinear', align_corners=False)
            pooled.append(upsampled)
        return torch.cat(pooled, dim=1)


class FeatMapNet(nn.Module):
    def __init__(self, scales=[1.0, 0.5, 0.25], pool_sizes=[1, 2, 4]):
        super().__init__()
        self.scales = scales
        
        # Load pretrained VGG16 features (blocks 1-5)
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        features = list(vgg.features.children())
        
        # Split into blocks for multi-scale processing
        self.block1 = nn.Sequential(*features[0:5])    # -> 64 channels
        self.block2 = nn.Sequential(*features[5:10])   # -> 128 channels
        self.block3 = nn.Sequential(*features[10:17])  # -> 256 channels
        self.block4 = nn.Sequential(*features[17:24])  # -> 512 channels
        self.block5 = nn.Sequential(*features[24:31])  # -> 512 channels
        
        # Block 6: Additional conv block
        self.block6 = nn.Sequential(
            nn.Conv2d(512, 512, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1),
            nn.ReLU(inplace=True),
        )
        
        # Spatial pyramid pooling
        self.spp = SpatialPyramidPooling(pool_sizes)
        
        # Feature dimension after SPP
        spp_mult = 1 + len(pool_sizes)
        self.out_channels = 512 * spp_mult
        
        # Fusion layer for multi-scale features
        self.fusion = nn.Conv2d(self.out_channels * len(scales), 512, 1)
    
    def forward_single_scale(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        x = self.block6(x)
        x = self.spp(x)
        return x
    
    def forward(self, x):
        B, C, H, W = x.shape
        multi_scale_feats = []
        
        target_h, target_w = H // 16, W // 16
        
        for scale in self.scales:
            if scale != 1.0:
                scaled_x = F.interpolate(x, scale_factor=scale, mode='bilinear', align_corners=False)
            else:
                scaled_x = x
            
            feat = self.forward_single_scale(scaled_x)
            
            if feat.shape[2:] != (target_h, target_w):
                feat = F.interpolate(feat, size=(target_h, target_w), mode='bilinear', align_corners=False)
            
            multi_scale_feats.append(feat)
        
        fused = torch.cat(multi_scale_feats, dim=1)
        out = self.fusion(fused)
        return out

class UnaryNet(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 256, 1),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.5),
            nn.Conv2d(256, num_classes, 1)
        )
    
    def forward(self, feat_map):
        return self.net(feat_map)

class PairwiseNet(nn.Module):
    def __init__(self, in_channels, num_classes, relation_type='surrounding'):
        super().__init__()
        self.num_classes = num_classes
        self.relation_type = relation_type
        
        self.edge_net = nn.Sequential(
            nn.Conv2d(in_channels * 2, 256, 1),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.5),
            nn.Conv2d(256, num_classes * num_classes, 1)
        )
    
    def get_shifted_features(self, feat, direction):
        B, C, H, W = feat.shape
        if direction == 'right':
            shifted = F.pad(feat[:, :, :, :-1], (1, 0, 0, 0))
        elif direction == 'left':
            shifted = F.pad(feat[:, :, :, 1:], (0, 1, 0, 0))
        elif direction == 'down':
            shifted = F.pad(feat[:, :, :-1, :], (0, 0, 1, 0))
        elif direction == 'up':
            shifted = F.pad(feat[:, :, 1:, :], (0, 0, 0, 1))
        return shifted
    
    def forward(self, feat_map):
        directions = ['right', 'left', 'down', 'up']
        pairwise_scores = []
        
        for d in directions:
            shifted = self.get_shifted_features(feat_map, d)
            edge_feat = torch.cat([feat_map, shifted], dim=1)
            scores = self.edge_net(edge_feat)
            pairwise_scores.append(scores)
        
        return torch.stack(pairwise_scores, dim=0).mean(dim=0)

class DeepCRFModel(nn.Module):
    def __init__(self, num_classes=33, use_pairwise=True):
        super().__init__()
        self.num_classes = num_classes
        self.use_pairwise = use_pairwise
        
        self.featmap_net = FeatMapNet(scales=[1.0, 0.5], pool_sizes=[1, 2])
        self.unary_net = UnaryNet(512, num_classes)
        
        if use_pairwise:
            self.pairwise_net = PairwiseNet(512, num_classes, 'surrounding')
        
        self.compat = nn.Parameter(torch.eye(num_classes) * -1.0)
    
    def forward(self, x, return_features=False):
        feat = self.featmap_net(x)
        unary = self.unary_net(feat)
        
        if return_features:
            return unary, feat
        return unary
    
    def get_pairwise_scores(self, feat):
        if self.use_pairwise:
            return self.pairwise_net(feat)
        return None

## Training and loss classes

In [ ]:
class PiecewiseUnaryLoss(nn.Module):
    def __init__(self, ignore_index=255, class_weights=None):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(ignore_index=ignore_index, weight=class_weights)
    
    def forward(self, unary_scores, labels):
        if unary_scores.shape[2:] != labels.shape[1:]:
            unary_scores = F.interpolate(unary_scores, size=labels.shape[1:], 
                                         mode='bilinear', align_corners=False)
        return self.ce(unary_scores, labels)


class PiecewisePairwiseLoss(nn.Module):
    def __init__(self, num_classes, ignore_index=255):
        super().__init__()
        self.num_classes = num_classes
        self.ignore_index = ignore_index
    
    def forward(self, pairwise_scores, labels):
        B, CC, H, W = pairwise_scores.shape
        C = self.num_classes
        
        # Resize labels to match pairwise feature map size
        if labels.shape[1:] != (H, W):
            labels_small = F.interpolate(labels.float().unsqueeze(1), size=(H, W), 
                                         mode='nearest').squeeze(1).long()
        else:
            labels_small = labels
        
        # Get shifted labels for horizontal and vertical neighbors
        # Horizontal: compare with right neighbor
        labels_center = labels_small[:, :, :-1]  # All except last column
        labels_right = labels_small[:, :, 1:]     # All except first column
        
        # Vertical: compare with bottom neighbor  
        labels_top = labels_small[:, :-1, :]      # All except last row
        labels_bottom = labels_small[:, 1:, :]    # All except first row
        
        # Create valid masks (both pixels must be valid, not ignore_index)
        valid_h = (labels_center != self.ignore_index) & (labels_right != self.ignore_index)
        valid_v = (labels_top != self.ignore_index) & (labels_bottom != self.ignore_index)
        
        # Flatten spatial dimensions and select valid pairs
        # Horizontal pairs
        B_h, H_h, W_h = labels_center.shape
        pairwise_h = pairwise_scores[:, :, :, :-1]  # Match spatial size with labels
        pairwise_h_flat = pairwise_h.reshape(B_h, CC, -1).permute(0, 2, 1)  # (B, H*W, C*C)
        
        # Vertical pairs
        B_v, H_v, W_v = labels_top.shape
        pairwise_v = pairwise_scores[:, :, :-1, :]  # Match spatial size with labels
        pairwise_v_flat = pairwise_v.reshape(B_v, CC, -1).permute(0, 2, 1)  # (B, H*W, C*C)
        
        # Compute targets: center_label * C + neighbor_label
        target_h = labels_center * C + labels_right
        target_v = labels_top * C + labels_bottom
        
        # Flatten everything
        pairwise_h_flat = pairwise_h_flat.reshape(-1, CC)
        pairwise_v_flat = pairwise_v_flat.reshape(-1, CC)
        target_h_flat = target_h.reshape(-1)
        target_v_flat = target_v.reshape(-1)
        valid_h_flat = valid_h.reshape(-1)
        valid_v_flat = valid_v.reshape(-1)
        
        # Compute loss only on valid pairs
        loss = 0
        count = 0
        
        if valid_h_flat.sum() > 0:
            loss += F.cross_entropy(pairwise_h_flat[valid_h_flat], target_h_flat[valid_h_flat])
            count += 1
        if valid_v_flat.sum() > 0:
            loss += F.cross_entropy(pairwise_v_flat[valid_v_flat], target_v_flat[valid_v_flat])
            count += 1
        
        return loss / max(count, 1)

## Inference and utility

In [ ]:
class MeanFieldInference(nn.Module):
    def __init__(self, num_classes, num_iterations=5):
        super().__init__()
        self.num_classes = num_classes
        self.num_iterations = num_iterations
        self.register_buffer('spatial_kernel', self._create_spatial_kernel())
    
    def _create_spatial_kernel(self, size=5):
        x = torch.arange(size) - size // 2
        kernel = torch.exp(-0.5 * (x.unsqueeze(0)**2 + x.unsqueeze(1)**2))
        kernel = kernel / kernel.sum()
        return kernel.view(1, 1, size, size)
    
    def forward(self, unary, pairwise_compat=None, image=None):
        B, C, H, W = unary.shape
        Q = F.softmax(unary, dim=1)
        
        for _ in range(self.num_iterations):
            Q_padded = F.pad(Q, (2, 2, 2, 2), mode='reflect')
            messages = F.conv2d(Q_padded.view(B*C, 1, H+4, W+4), 
                               self.spatial_kernel).view(B, C, H, W)
            
            if pairwise_compat is not None:
                messages = messages.permute(0, 2, 3, 1)
                messages = torch.matmul(messages, pairwise_compat)
                messages = messages.permute(0, 3, 1, 2)
            
            Q = F.softmax(unary - messages, dim=1)
        
        return Q

def compute_class_weights(dataset, num_classes=33, num_samples=100):
    print("Computing class weights from dataset")
    class_counts = torch.zeros(num_classes)
    
    for i in range(min(num_samples, len(dataset))):
        _, label = dataset[i]
        for c in range(num_classes):
            class_counts[c] += (label == c).sum().item()
    
    # Inverse frequency weighting
    total = class_counts.sum()
    class_weights = total / (num_classes * class_counts + 1e-8)
    
    # Normalize and clip to reasonable range
    class_weights = class_weights / class_weights.mean()
    class_weights = torch.clamp(class_weights, 0.1, 10.0)
    return class_weights

## Training

In [ ]:
def train_piecewise(model, loader, optimizer, unary_loss_fn, pairwise_loss_fn, 
                    device, train_pairwise=True):
    model.train()
    total_unary_loss = 0
    total_pairwise_loss = 0
    batch_losses = []
    
    for imgs, labels in tqdm(loader, desc="Training"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        
        unary, feat = model(imgs, return_features=True)
        
        loss_u = unary_loss_fn(unary, labels)
        total_unary_loss += loss_u.item()
        batch_losses.append(loss_u.item())
        
        loss_p = torch.tensor(0.0, device=device)
        if train_pairwise and model.use_pairwise:
            pairwise = model.get_pairwise_scores(feat)
            loss_p = pairwise_loss_fn(pairwise, labels)
            total_pairwise_loss += loss_p.item()
        
        loss = loss_u + 0.5 * loss_p
        loss.backward()
        optimizer.step()
    
    n = len(loader)
    avg_loss = total_unary_loss / n
    
    return avg_loss, total_pairwise_loss / n

def save_checkpoint(model, optimizer, epoch, filename):
    checkpoint_dir = "checkpoints"
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)
    
    save_path = os.path.join(checkpoint_dir, filename)
    
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, save_path)
    print(f"Saved checkpoint: {save_path}")

## Evaluation and prediction

In [ ]:
def evaluate_all_stages(device, mat_path="sift_flow.mat"):
    print(f"\n Starting Multi-Stage Evaluation on {device}...")
    
    # 1. Setup Data
    test_tf = TestAugmentation(size=(512, 512))
    to_tensor = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    if not os.path.exists(mat_path):
        print("Dataset not found.")
        return

    test_ds = MatSiftDataset(mat_path, 'test', to_tensor, test_tf)
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)
    
    # 2. Define Stages Configuration
    stages = [
        ("Stage 1 (Unary)",    "checkpoints/sift_flow_stage1_logs.pth", False, False),
        ("Stage 2 (Joint)",    "checkpoints/sift_flow_stage2_logs.pth", True,  True),
        ("Stage 3 (Fine-tune)","checkpoints/sift_flow_stage3_logs.pth", True,  True)
    ]
    
    results = []
    
    # 3. Loop through stages
    for name, path, use_pw, use_crf in stages:
        if not os.path.exists(path):
            print(f"Skipping {name}: {path} not found.")
            continue
            
        print(f"\nEvaluating {name}...")
        
        # Initialize correct architecture for the stage
        model = DeepCRFModel(num_classes=33, use_pairwise=use_pw).to(device)
        
        # Load Weights
        checkpoint = torch.load(path, map_location=device)
        
        # Handle dict wrapping logic
        state_dict = checkpoint
        if 'model' in checkpoint: state_dict = checkpoint['model']
        elif 'model_state_dict' in checkpoint: state_dict = checkpoint['model_state_dict']
            
        # Load
        model.load_state_dict(state_dict, strict=True)
        
        # Run Metrics
        pix, macc, miou = evaluate_detailed(model, test_loader, device, use_crf=use_crf, num_classes=33)
        
        print(f"  ->Global Acc: {pix:.4f}")
        print(f"  -> Mean IoU:   {miou:.4f}")
        
        results.append({
            "Stage": name,
            "Pixel Acc": pix,
            "Mean Acc": macc,
            "Mean IoU": miou
        })

    # 4. Display Final Comparison Table
    print("STAGE COMPARISON REPORT")
    df = pd.DataFrame(results)
    print(df.to_string(index=False))


def unnormalize(tensor):
    # Reverses the ImageNet normalization to display the image correctly
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    
    img = tensor.cpu().clone()
    img = img * std + mean  # Un-normalize
    img = torch.clamp(img, 0, 1)
    return img.permute(1, 2, 0).numpy() # (H, W, C)

def colorize_mask(mask, num_classes=33):
    # Applies a distinct colormap to the segmentation mask
    # Create a distinctive colormap (nipy_spectral is good for many classes)
    cmap = plt.get_cmap('nipy_spectral')
    
    # Normalize mask 0-1 for the colormap lookup
    mask_norm = mask.astype(float) / num_classes
    
    # Get RGB values from colormap
    colored_mask = cmap(mask_norm)[:, :, :3] # Drop Alpha channel
    
    # Handle 'ignore' index (255) -> Make it Black
    colored_mask[mask == 255] = [0, 0, 0] 
    
    return colored_mask

def visualize_predictions(model, dataset, device, num_samples=3):
    model.eval()
    
    # Select random indices
    indices = np.random.choice(len(dataset), num_samples, replace=False)
    
    # Create subplots
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples))
    
    # Setup CRF
    mf = MeanFieldInference(num_classes=33, num_iterations=10).to(device)
    
    for i, idx in enumerate(indices):
        img_tensor, label_tensor = dataset[idx]
        
        # Prepare input
        img_input = img_tensor.unsqueeze(0).to(device) # Add batch dim
        
        with torch.no_grad():
            # 1. Unary Potentials
            unary = model(img_input)
            
            # 2. Resize if needed
            if unary.shape[2:] != label_tensor.shape:
                unary = F.interpolate(unary, size=label_tensor.shape, 
                                      mode='bilinear', align_corners=False)
            
            # 3. CRF Inference (Refinement)
            probs = mf(unary, model.compat)
            pred_mask = probs.argmax(dim=1).cpu().squeeze().numpy()
        
        # Prepare visualization data
        original_img = unnormalize(img_tensor)
        gt_mask = label_tensor.numpy()
        
        # Colorize
        gt_color = colorize_mask(gt_mask)
        pred_color = colorize_mask(pred_mask)
        
        # PLOTTING
        if num_samples == 1: row_axes = axes
        else: row_axes = axes[i]
        
        # 1. Original Image
        row_axes[0].imshow(original_img)
        row_axes[0].set_title(f"Input Image (Idx: {idx})")
        row_axes[0].axis('off')
        
        # 2. Ground Truth
        row_axes[1].imshow(gt_color)
        row_axes[1].set_title("Ground Truth")
        row_axes[1].axis('off')
        
        # 3. Prediction
        row_axes[2].imshow(pred_color)
        row_axes[2].set_title("Prediction (CRF)")
        row_axes[2].axis('off')
        
    plt.tight_layout()
    plt.savefig("prediction_samples.png")
    print("Saved visualization to 'prediction_samples.png'")
    plt.show()

## Main

In [ ]:
def main():
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Device: {device}")
    
    # CONFIG
    MAT_FILE_PATH = "sift_flow.mat" 
    NUM_CLASSES = 33
    BATCH_SIZE = 4
    LR_PRETRAINED = 1e-4
    LR_NEW = 1e-3
    
    # EPOCHS
    EPOCHS_STAGE1 = 40
    EPOCHS_STAGE2 = 10
    EPOCHS_STAGE3 = 5
    
    # DATA LOADING
    train_augmentation = SegmentationAugmentation(base_size=512, crop_size=480, scale_range=(0.7, 1.3))
    test_augmentation = TestAugmentation(size=(512, 512))
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    if not os.path.exists(MAT_FILE_PATH):
        print(f"ERROR: {MAT_FILE_PATH} not found.")
        return

    train_ds = MatSiftDataset(MAT_FILE_PATH, split='train', transform=transform, augmentation=train_augmentation)
    test_ds = MatSiftDataset(MAT_FILE_PATH, split='test', transform=transform, augmentation=test_augmentation)
    
    verify_dataset_range(train_ds, "Train")
    class_weights = compute_class_weights(train_ds, NUM_CLASSES).to(device)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # HISTORY TRACKER
    history = {
        'stage1': {'unary_loss': [], 'pixel_acc': [], 'mean_acc': [], 'mean_iou': []},
        'stage2': {'unary_loss': [], 'pairwise_loss': [], 'pixel_acc': [], 'mean_acc': [], 'mean_iou': []},
        'stage3': {'unary_loss': [], 'pairwise_loss': [], 'pixel_acc': [], 'mean_acc': [], 'mean_iou': []}
    }
    
    # STAGE 1
    print("\n" + "="*60 + "\nStage 1: Training Unary Potentials\n" + "="*60)
    model = DeepCRFModel(num_classes=NUM_CLASSES, use_pairwise=False).to(device)
    
    # Parameters (Using original architecture)
    pretrained_params = list(model.featmap_net.block1.parameters()) + \
                        list(model.featmap_net.block2.parameters()) + \
                        list(model.featmap_net.block3.parameters()) + \
                        list(model.featmap_net.block4.parameters()) + \
                        list(model.featmap_net.block5.parameters())
    new_params = list(model.featmap_net.block6.parameters()) + \
                 list(model.featmap_net.fusion.parameters()) + \
                 list(model.unary_net.parameters())
    
    optimizer = torch.optim.SGD([
        {'params': pretrained_params, 'lr': LR_PRETRAINED},
        {'params': new_params, 'lr': LR_NEW}
    ], momentum=0.9, weight_decay=5e-4)
    
    unary_loss_fn = PiecewiseUnaryLoss(class_weights=class_weights)
    pairwise_loss_fn = PiecewisePairwiseLoss(NUM_CLASSES)
    
    for epoch in range(EPOCHS_STAGE1):
        loss_u, _ = train_piecewise(model, train_loader, optimizer, unary_loss_fn, pairwise_loss_fn, device, train_pairwise=False)
        p_acc, m_acc, m_iou = evaluate_detailed(model, test_loader, device, num_classes=NUM_CLASSES)
        
        # Log
        history['stage1']['unary_loss'].append(loss_u)
        history['stage1']['pixel_acc'].append(p_acc)
        history['stage1']['mean_acc'].append(m_acc)
        history['stage1']['mean_iou'].append(m_iou)
        
        print(f"[Stage1] Epoch {epoch+1} Loss: {loss_u:.4f} | PixAcc: {p_acc:.4f} mAcc: {m_acc:.4f} mIoU: {m_iou:.4f}")
    
    # Save Stage 1
    save_checkpoint(model, optimizer, EPOCHS_STAGE1, "sift_flow_stage1_logs.pth")
    with open("training_log.json", "w") as f: json.dump(history, f)

    # STAGE 2
    print("\n" + "="*60 + "\nStage 2: Joint Training (Pairwise)\n" + "="*60)
    model.use_pairwise = True
    model.pairwise_net = PairwiseNet(512, NUM_CLASSES).to(device)
    optimizer.add_param_group({'params': model.pairwise_net.parameters(), 'lr': LR_NEW})
    
    for epoch in range(EPOCHS_STAGE2):
        loss_u, loss_p = train_piecewise(model, train_loader, optimizer, unary_loss_fn, pairwise_loss_fn, device, train_pairwise=True)
        p_acc, m_acc, m_iou = evaluate_detailed(model, test_loader, device, use_crf=True, num_classes=NUM_CLASSES)
        
        history['stage2']['unary_loss'].append(loss_u)
        history['stage2']['pairwise_loss'].append(loss_p)
        history['stage2']['pixel_acc'].append(p_acc)
        history['stage2']['mean_acc'].append(m_acc)
        history['stage2']['mean_iou'].append(m_iou)
        
        print(f"[Stage2] Epoch {epoch+1} Loss U:{loss_u:.4f} P:{loss_p:.4f} | PixAcc: {p_acc:.4f} mAcc: {m_acc:.4f} mIoU: {m_iou:.4f}")

    # Save Stage 2
    save_checkpoint(model, optimizer, EPOCHS_STAGE2, "sift_flow_stage2_logs.pth")
    with open("training_log.json", "w") as f: json.dump(history, f)

    # STAGE 3
    print("\n" + "="*60 + "\nStage 3: Fine-tuning (Freeze Backbone)\n" + "="*60)
    for param in model.featmap_net.parameters(): param.requires_grad = False
    
    optimizer = torch.optim.SGD(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_NEW, momentum=0.9)
    
    for epoch in range(EPOCHS_STAGE3):
        loss_u, loss_p = train_piecewise(model, train_loader, optimizer, unary_loss_fn, pairwise_loss_fn, device, train_pairwise=True)
        p_acc, m_acc, m_iou = evaluate_detailed(model, test_loader, device, use_crf=True, num_classes=NUM_CLASSES)
        
        history['stage3']['unary_loss'].append(loss_u)
        history['stage3']['pairwise_loss'].append(loss_p)
        history['stage3']['pixel_acc'].append(p_acc)
        history['stage3']['mean_acc'].append(m_acc)
        history['stage3']['mean_iou'].append(m_iou)
        
        print(f"[Stage3] Epoch {epoch+1} Loss U:{loss_u:.4f} P:{loss_p:.4f} | PixAcc: {p_acc:.4f} mAcc: {m_acc:.4f} mIoU: {m_iou:.4f}")

    save_checkpoint(model, optimizer, EPOCHS_STAGE3, "sift_flow_stage3_logs.pth")
    with open("training_log.json", "w") as f: json.dump(history, f)
    
    print("\nTraining Complete, logs saved to 'training_log.json'")

In [ ]:
if __name__ == "__main__":
    main()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    evaluate_all_stages(device)    

In [ ]:
# Prediction visualisation

if __name__ == "__main__":
    # Settings
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    MAT_PATH = "sift_flow.mat"
    CHECKPOINT_PATH = "checkpoints/sift_flow_stage3_logs.pth"
    
    # Re-create Dataset
    test_tf = TestAugmentation(size=(512, 512))
    to_tensor = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    if os.path.exists(MAT_PATH):
        test_ds = MatSiftDataset(MAT_PATH, 'test', to_tensor, test_tf)
        
        # Re-load Model
        model = DeepCRFModel(num_classes=33, use_pairwise=True).to(device)
        
        if os.path.exists(CHECKPOINT_PATH):
            print(f"Loading weights from {CHECKPOINT_PATH}...")
            ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
            
            # Handle dictionary mismatch
            if 'model' in ckpt: model.load_state_dict(ckpt['model'])
            elif 'model_state_dict' in ckpt: model.load_state_dict(ckpt['model_state_dict'])
            else: model.load_state_dict(ckpt)
            
            # RUN VISUALIZATION
            visualize_predictions(model, test_ds, device, num_samples=3)
            
        else:
            print("Checkpoint not found.")
    else:
        print("Dataset not found.")